# Filter Best Models & Analyze Overlap
1. **Filter Best Models**: Select top 100 performing protein models (MSE) for each validation dataset.
2. **Overlap Analysis**: Identify proteins shared between the training set (CCLE) and validation datasets (TMT, iTRAQ, LabelFree).


In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import mean_squared_error
import tables

# Adjust display
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 20)


In [2]:
# Path to results
h5_file = 'res_data/enet_all_results.h5'

print(f"Loading results from {h5_file}...")
if os.path.exists(h5_file):
    with pd.HDFStore(h5_file) as store:
        keys = store.keys()
        df_list = [store[key] for key in keys]
        combined_df = pd.concat(df_list, ignore_index=True)
    print(f"Loaded combined_df. Shape: {combined_df.shape}")
else:
    print(f"ERROR: File {h5_file} not found!")
    combined_df = pd.DataFrame()


Loading results from res_data/enet_all_results.h5...
Loaded combined_df. Shape: (1886697, 5)


In [3]:
if not combined_df.empty:
    print("Calculating MSE per protein...")
    metrics_df = combined_df.groupby('protein').apply(
        lambda x: pd.Series({
            'mse': mean_squared_error(x['true'], x['predicted']),
            'count': len(x)
        })
    ).reset_index()
    
    # These are effectively the CCLE proteins we have models for
    ccle_proteins = set(metrics_df['protein'].unique())
    print(f"Total CCLE Proteins (with models): {len(ccle_proteins)}")
else:
    metrics_df = pd.DataFrame()
    ccle_proteins = set()


Calculating MSE per protein...
Total CCLE Proteins (with models): 5113


C:\Users\hp\AppData\Local\Temp\ipykernel_33120\3016947553.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  metrics_df = combined_df.groupby('protein').apply(


In [4]:
# Path to feature lists
LISTS_DIR = "../processed_data/feature_lists"
datasets = ["TMT", "iTRAQ", "LabelFree"]
N_TOP = 100

validation_sets = {} # Store sets for overlap analysis

if not metrics_df.empty:
    for group in datasets:
        print(f"\n=== Processing {group} Dataset (MSE Filtering) ===")
        list_path = os.path.join(LISTS_DIR, f"{group}_proteins.txt")
        
        if os.path.exists(list_path):
            with open(list_path, 'r') as f:
                available_prots = [line.strip() for line in f if line.strip()]
            
            # Store for later overlap analysis
            validation_sets[group] = set(available_prots)
            print(f"  Available proteins in {group}: {len(available_prots)}")
            
            # Filter MSE results
            group_df = metrics_df[metrics_df['protein'].isin(available_prots)].copy()
            
            # Sort and Take Top N
            top_df = group_df.sort_values(by='mse', ascending=True).head(N_TOP)
            
            if not top_df.empty:
                # Save as .txt
                out_txt = f"{group}_best_{N_TOP}_proteins.txt"
                with open(out_txt, 'w') as f_out:
                    for p in top_df['protein']:
                        f_out.write(f"{p}\n")
                print(f"  Saved Top {len(top_df)} {group} proteins (by MSE) to: {out_txt}")
            else:
                print("  No proteins matched.")
        else:
            print(f"  WARNING: Protein list file not found: {list_path}")
            validation_sets[group] = set()
else:
    print("Metrics dataframe is empty.")



=== Processing TMT Dataset (MSE Filtering) ===
  Available proteins in TMT: 6885
  Saved Top 100 TMT proteins (by MSE) to: TMT_best_100_proteins.txt

=== Processing iTRAQ Dataset (MSE Filtering) ===
  Available proteins in iTRAQ: 4994
  Saved Top 100 iTRAQ proteins (by MSE) to: iTRAQ_best_100_proteins.txt

=== Processing LabelFree Dataset (MSE Filtering) ===
  Available proteins in LabelFree: 2563
  Saved Top 100 LabelFree proteins (by MSE) to: LabelFree_best_100_proteins.txt


In [5]:
print("\n=== CCLE Overlap Analysis ===")

# 1. CCLE vs Each Dataset
for group in datasets:
    val_set = validation_sets.get(group, set())
    overlap = ccle_proteins.intersection(val_set)
    print(f"\nCCLE ({len(ccle_proteins)}) vs {group} ({len(val_set)}):")
    print(f"  Overlap Count: {len(overlap)}")
    
    if len(overlap) > 0:
        out_name = f"CCLE_vs_{group}_overlap.txt"
        with open(out_name, 'w') as f:
            for p in sorted(list(overlap)):
                f.write(f"{p}\n")
        print(f"  Saved list to: {out_name}")

# 2. Intersection of ALL 4 (CCLE + TMT + iTRAQ + LabelFree)
# Start with CCLE
common_all = ccle_proteins.copy()
for group in datasets:
    common_all = common_all.intersection(validation_sets.get(group, set()))

print(f"\nProteins Common to ALL 4 Datasets (CCLE + TMT + iTRAQ + LabelFree):")
print(f"  Count: {len(common_all)}")

if len(common_all) > 0:
    out_all = "Common_All_4_Datasets.txt"
    with open(out_all, 'w') as f:
        for p in sorted(list(common_all)):
            f.write(f"{p}\n")
    print(f"  Saved list to: {out_all}")
    print("  Proteins:", sorted(list(common_all)))



=== CCLE Overlap Analysis ===

CCLE (5113) vs TMT (6885):
  Overlap Count: 4455
  Saved list to: CCLE_vs_TMT_overlap.txt

CCLE (5113) vs iTRAQ (4994):
  Overlap Count: 3730
  Saved list to: CCLE_vs_iTRAQ_overlap.txt

CCLE (5113) vs LabelFree (2563):
  Overlap Count: 2176
  Saved list to: CCLE_vs_LabelFree_overlap.txt

Proteins Common to ALL 4 Datasets (CCLE + TMT + iTRAQ + LabelFree):
  Count: 2157
  Saved list to: Common_All_4_Datasets.txt
  Proteins: ['A2M', 'AARS2', 'AASDHPPT', 'ABCB7', 'ABCD3', 'ABCE1', 'ABCF1', 'ABHD10', 'ABHD11', 'ABHD12', 'ABHD14B', 'ABHD16A', 'ABI1', 'ACAA1', 'ACAA2', 'ACAD10', 'ACAD8', 'ACAD9', 'ACADM', 'ACADS', 'ACADSB', 'ACADVL', 'ACAT1', 'ACAT2', 'ACBD3', 'ACIN1', 'ACLY', 'ACO1', 'ACO2', 'ACOT13', 'ACOT7', 'ACOT9', 'ACOX1', 'ACP1', 'ACP2', 'ACSF2', 'ACSF3', 'ACSL1', 'ACSL3', 'ACSL4', 'ACSL5', 'ACTB', 'ACTL6A', 'ACTN1', 'ACTN4', 'ACTR10', 'ACTR1A', 'ACTR2', 'ACTR3', 'ACY1', 'ADAM10', 'ADAR', 'ADD1', 'ADD3', 'ADH5', 'ADK', 'ADPGK', 'ADRM1', 'ADSL', 'AFG3L2',